In [ ]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

### Importing Library

In [ ]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF Python module provided by PyMuPDF.
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
DPO patch applied.
GPU: Tesla T4


### Loading Dataset

In [ ]:
pdf_path = "/content/HR-Policy.pdf"
doc = fitz.open(pdf_path)
print(f"Total Pages in PDF: {len(doc)}")



Total Pages in PDF: 18


In [ ]:
import re

def clean_text(text: str) -> str:
    # 1. Normalize heading markers first to avoid interference with newline processing
    text = re.sub(r"={2,}.*?={2,}", lambda m: m.group(0).replace("=", "").strip() + ".", text)

    # 2. Normalize two or more newlines (possibly with spaces in between) to a unique temporary marker
    # This step tries to identify true paragraph breaks (e.g., \n\n, \n \n, \n\n\n).
    text = re.sub(r"(\n\s*){2,}", " [PARAGRAPH_BREAK] ", text)

    # 3. Replace all remaining single newlines with a space (these are assumed to be line breaks within a paragraph)
    text = text.replace("\n", " ")

    # 4. Normalize spaces: collapse multiple spaces into a single space
    text = re.sub(r"[ \t]+", " ", text)

    # 5. Restore the paragraph breaks from the temporary marker to double newlines
    text = text.replace(" [PARAGRAPH_BREAK] ", "\n\n")

    return text.strip()

# Extract text from the PDF document
full_text = ""
for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    full_text += page.get_text()

cleaned = clean_text(full_text)

# Split into paragraphs (blank-line separated)
paragraphs = [p.strip() for p in cleaned.split("\n\n") if len(p.strip()) > 40]
print(f"Number of paragraphs after cleaning: {len(paragraphs)}")
print(paragraphs[0])

Number of paragraphs after cleaning: 78
RIKALP CAPITAL PRIVATE LIMITED (FORMERLY KNOWN AS SETHI SECURITIES PRIVATE LIMITED)


### Chunking

In [ ]:
def chunk_paragraphs(paragraphs, min_chars=400, max_chars=800):
    chunks = []
    buffer = ""
    for p in paragraphs:
        if len(buffer) + len(p) < max_chars:
            buffer = (buffer + "\n\n" + p).strip()
        else:
            if buffer:
                chunks.append(buffer)
            buffer = p
        if len(buffer) >= min_chars:
            chunks.append(buffer)
            buffer = ""
    if buffer:
        chunks.append(buffer)
    return chunks

paragraphs = chunk_paragraphs(paragraphs)
print(f"Number of training chunks: {len(paragraphs)}")
print("\n--- Sample chunk ---\n")
print(paragraphs[0])

Number of training chunks: 43

--- Sample chunk ---

RIKALP CAPITAL PRIVATE LIMITED (FORMERLY KNOWN AS SETHI SECURITIES PRIVATE LIMITED)

REGISTERED OFFICE EAST INDIA HOUSE 20B ABDUL HAMID STREET, 5TH FLOOR, ROOM NO. 5A-1, ESPLANADE, KOLKATA, WEST BENGAL, INDIA 700069

CORPORATE OFFICE PLOT NO. 42, SECOND FLOOR, MONIKA VIHAR, MANYAWAS, MANSAROVAR, JAIPUR, RAJASTHAN, INDIA 302020


In [ ]:
# Create the Data Folder
from pathlib import Path

# Create The data Folder
Path("data").mkdir(exist_ok=True)

print("Data folder created successfully")



## Save the Paragraphs

output_file = "data/non_instruction_data.txt"

with open(output_file, "w", encoding="utf-8") as file:
    for paragraph in paragraphs:
        file.write(paragraph + "\n\n")

print("Dataset saved successfully!")



### Verify the Saved Dataset
with open("data/non_instruction_data.txt", "r", encoding="utf-8") as file:
    saved_text = file.read()



# Check Dataset Statistics

print("=" * 50)
print("Dataset Statistics")
print("=" * 50)

print(f"Total Paragraphs : {len(paragraphs)}")


Data folder created successfully
Dataset saved successfully!
Dataset Statistics
Total Paragraphs : 43


### Converting Dataset to JSONL

In [ ]:
# -------------------------
# Convert the paragraph dataset into JSONL format
# Each line is a standalone JSON object: {"text": "<paragraph>"}
# This is the standard format expected by HuggingFace `datasets`
# (load_dataset("json", ...)) for non-instruction / raw-text fine-tuning.
# -------------------------
import json
from pathlib import Path

Path("data").mkdir(exist_ok=True)

jsonl_output_file = "data/non_instruction_data.jsonl"

with open(jsonl_output_file, "w", encoding="utf-8") as f:
    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        record = {"text": paragraph}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"JSONL dataset saved successfully to {jsonl_output_file}")

# -------------------------
# Verify the saved JSONL file
# -------------------------
records = []
with open(jsonl_output_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))

print("=" * 50)
print("JSONL Dataset Statistics")
print("=" * 50)
print(f"Total records : {len(records)}")
print(f"Avg chars/record : {sum(len(r['text']) for r in records) / len(records):.1f}")
print("\n--- Sample record ---\n")
print(json.dumps(records[0], indent=2, ensure_ascii=False))


JSONL dataset saved successfully to data/non_instruction_data.jsonl
JSONL Dataset Statistics
Total records : 43
Avg chars/record : 620.2

--- Sample record ---

{
  "text": "RIKALP CAPITAL PRIVATE LIMITED (FORMERLY KNOWN AS SETHI SECURITIES PRIVATE LIMITED)\n\nREGISTERED OFFICE EAST INDIA HOUSE 20B ABDUL HAMID STREET, 5TH FLOOR, ROOM NO. 5A-1, ESPLANADE, KOLKATA, WEST BENGAL, INDIA 700069\n\nCORPORATE OFFICE PLOT NO. 42, SECOND FLOOR, MONIKA VIHAR, MANYAWAS, MANSAROVAR, JAIPUR, RAJASTHAN, INDIA 302020"
}


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "data/non_instruction_data.jsonl"},
)

# Define a tokenization function
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length", # Explicitly pad to max_seq_length
        return_tensors="pt"
    )
    # Ensure input_ids are LongTensor explicitly
    tokenized["input_ids"] = tokenized["input_ids"].long()
    return tokenized

# Apply the tokenization to the dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"], # Remove the original text column
    num_proc=0, # Process in a single thread to avoid multiprocessing issues
)

# SFTTrainer expects 'labels' to be present. For language modeling, labels are usually input_ids.
tokenized_dataset = tokenized_dataset.map(lambda examples: {'labels': examples['input_ids']}, batched=True)


print("Original dataset:", dataset)
print("Tokenized dataset:", tokenized_dataset)


Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Original dataset: DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 43
    })
})
Tokenized dataset: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 43
    })
})


### Loading Base Model

In [ ]:

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048 # The model can look at 2048 tokens at one time.

# Automatically detect data type
dtype = None # These represent how numbers are stored inside the GPU

# Load model in 4-bit quantization
load_in_4bit = True

# Load the Pretrained model and tokenizer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)





==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "Employees are eligible for"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print("BEFORE fine-tuning:\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))

BEFORE fine-tuning:

Employees are eligible for a 10% discount on their annual health insurance premium if they participate in the company's wellness program. If an employee who is not part of the program pays $25 per month, how much will they save annually by participating in the program? To determine how much an employee saves annually by


### Fine-tuning the Model

Now, we will fine-tune the pre-trained `Qwen2.5-1.5B-Instruct-bnb-4bit` model using the HR policy paragraphs we extracted. We will use the `SFTTrainer` from the `trl` library for this. The `SFTTrainer` allows us to efficiently train the model on our custom dataset.

### Applying Qlora

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16, #It controls how much the model is allowed to learn through the LoRA layers.
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",

    ],

    lora_alpha = 16,
    lora_dropout = 0.05, #dropout randomly ignores some neurons during training.
    bias = "none",
    use_gradient_checkpointing="unsloth",
    # Gradient checkpointing reduces GPU memory usage by recomputing intermediate activations during backpropagation instead of storing them all. This allows us to train larger models on limited GPU memory.
    random_state = 3047 # make the training reproducible.
)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


### Training On Raw Dataset

In [ ]:

from transformers import TrainingArguments
from trl import SFTTrainer

training_args =  TrainingArguments(
        output_dir = "outputs_non_instruction",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",
        save_strategy = "no",
    )

### Creating Trainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset['train'], # Use the pre-tokenized dataset
    # dataset_text_field = "text", # REMOVE THIS LINE as data is already tokenized
    max_seq_length = max_seq_length, # Still needed for some internal checks like padding in data collator
    dataset_num_proc = 0, # Changed from 2 to 0 to disable multiprocessing for dataset processing
    packing = False, # Set packing to False to troubleshoot the RuntimeError
    args = training_args
)

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 43 | Num Epochs = 3 | Total steps = 18
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,14.686300
10,13.119800
15,9.650200


TrainOutput(global_step=18, training_loss=11.81607288784451, metrics={'train_runtime': 201.8364, 'train_samples_per_second': 0.639, 'train_steps_per_second': 0.089, 'total_flos': 2106358499377152.0, 'train_loss': 11.81607288784451, 'epoch': 3.0})


In [ ]:
from pathlib import Path

# Create the 'model' directory if it doesn't exist
Path("model").mkdir(exist_ok=True)

print("Directory 'model' created successfully.")

Directory 'model' created successfully.


In [ ]:
# Save the LoRA adapter
model.save_pretrained("model/non_instruction_self_hr_policy_adapter")

# Save the tokenizer
tokenizer.save_pretrained("model/non_instruction_self_hr_policy_tokenizer")

('model/non_instruction_self_hr_policy_tokenizer/tokenizer_config.json',
 'model/non_instruction_self_hr_policy_tokenizer/special_tokens_map.json',
 'model/non_instruction_self_hr_policy_tokenizer/chat_template.jinja',
 'model/non_instruction_self_hr_policy_tokenizer/vocab.json',
 'model/non_instruction_self_hr_policy_tokenizer/merges.txt',
 'model/non_instruction_self_hr_policy_tokenizer/added_tokens.json',
 'model/non_instruction_self_hr_policy_tokenizer/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Define the base path within your Google Drive
drive_save_path = "/content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter"

# Create the directory if it doesn't exist
os.makedirs(drive_save_path, exist_ok=True)

print(f"Google Drive save path created: {drive_save_path}")

Google Drive save path created: /content/drive/MyDrive/GEN_AI_FINE_TUNING/Hugging_FaceVsUnsolth/Project/Adapter


In [ ]:
# Save the LoRA adapter to Google Drive
model.save_pretrained(os.path.join(drive_save_path, "non_instruction_self_hr_policy_adapter"))

# Save the tokenizer to Google Drive
tokenizer.save_pretrained(os.path.join(drive_save_path, "non_instruction_self_hr_policy_tokenizer"))

print("Model and tokenizer successfully saved to Google Drive!")

Model and tokenizer successfully saved to Google Drive!


## Pushing To Hugging Face

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get("HF_Token"))

In [ ]:
model.push_to_hub("Toji619/non_instruction_self_hr_policy_adapter")
tokenizer.push_to_hub("Toji619/non_instruction_self_hr_policy_tokenizer")

README.md:   0%|          | 0.00/565 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Saved model to https://huggingface.co/Toji619/non_instruction_self_hr_policy_adapter


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._tokenizer/tokenizer.json:  68%|######8   | 7.78MB / 11.4MB            

### Loading Adapter Again

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Toji619/non_instruction_self_hr_policy_adapter",
    max_seq_length = 2048,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "What are Conflict of Interest in company",
    "Maternity Leaves regarding policy",

]

for p in test_prompts:
    inputs = tokenizer(p, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
    print("PROMPT:", p)
    print("AFTER Stage 1 fine-tuning:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))
    print("-" * 80)

PROMPT: What are Conflict of Interest in company
AFTER Stage 1 fine-tuning:
What are Conflict of Interest in company?

Conflict of interest is a situation where an individual has a personal or financial interest that could influence their decision-making. In the context of companies, it can refer to situations where employees have a personal or financial interest that could affect their ability to make fair and impartial decisions.

Some examples of conflict of interest
--------------------------------------------------------------------------------
PROMPT: Maternity Leaves regarding policy
AFTER Stage 1 fine-tuning:
Maternity Leaves regarding policy and procedure

The maternity leave is a right of every woman who has been employed by the company. The employee must be on full-time employment for at least 6 months before she can take her maternity leave.

In order to ensure that all employees are aware of their rights, it is important that they
------------------------------------------